## 🎯 Learning Objectives
* Containerize a complex LangGraph agent pipeline using Docker.
* Deploy a LangGraph agent as a web service using modern frameworks like FastAPI.
* Integrate advanced monitoring and observability tools (e.g., LangSmith, custom metrics) into an agent pipeline.
* Analyze agent traces and metrics to debug and optimize deployed multi-agent systems.
* Design and implement a robust API for interacting with an advanced AI agent.


## Exercise: Deploy and Monitor an Advanced Agent Pipeline

**Lesson ID:** ADV01-L13

### Task Overview

In this exercise, you will take a pre-defined complex LangGraph agent and prepare it for production deployment. This involves containerization, exposing it as a web service, and implementing robust monitoring and tracing capabilities. The goal is to ensure that your advanced agent, potentially leveraging subgraphs, supervisor nodes, and nested composition, can be reliably operated and observed in a production-like environment.

### Background

As Senior AI engineers, building sophisticated multi-agent systems is only half the battle. The other, equally critical half, is ensuring these systems are deployable, performant, and observable in real-world scenarios. This exercise bridges the gap between development and operations, focusing on the practical aspects of bringing a LangGraph agent to life outside your local development environment.

### Requirements

1.  **Agent Definition**: You will be provided with a mock complex LangGraph agent. Your task is to deploy *this* agent.
2.  **Containerization**: Create a `Dockerfile` to containerize the agent and its dependencies. The container should be self-contained and runnable.
3.  **Web Service API**: Expose the agent's functionality via a RESTful API using `FastAPI`. The API should have at least one endpoint to receive a user query and return the agent's response.
4.  **Deployment Simulation**: While actual cloud deployment is beyond the scope of this single exercise, you should set up the environment such that the container can be run locally (e.g., using `uvicorn`) and simulate a production deployment.
5.  **Monitoring**: Implement basic monitoring for the agent. This should include:
    *   **Latency**: Measure the time taken for each agent run.
    *   **Token Usage (Conceptual)**: While we won't integrate with a real LLM for this exercise, conceptually track or log 'token usage' for each agent step.
    *   **Agent State Transitions**: Log significant state changes or node activations within the LangGraph execution.
6.  **Tracing**: Integrate with `LangSmith` (or a similar OpenTelemetry-compatible tracing system) to capture detailed traces of each agent run. This is crucial for time-travel debugging and understanding complex agent behavior.
7.  **Client Interaction**: Provide a simple Python client script or `curl` commands to interact with your deployed agent API.

### Evaluation Criteria

*   **Correctness of `Dockerfile`**: The container builds successfully and runs the application.
*   **API Functionality**: The FastAPI endpoint correctly receives input, processes it with the LangGraph agent, and returns a meaningful response.
*   **Monitoring Implementation**: Latency, conceptual token usage, and state transitions are logged or tracked effectively.
*   **LangSmith Integration**: Agent runs are successfully traced and visible in LangSmith (or a mock tracing output).
*   **Code Quality**: The solution is well-structured, readable, and includes appropriate comments.
*   **Robustness**: The API handles basic errors gracefully (e.g., invalid input).

Let's get started!


In [ ]:
import os
import time
import logging
from typing import List, Tuple, Dict, TypedDict, Union

# Mock LangChain/LangGraph components
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END, START
from langgraph.checkpoint.memory import MemorySaver

# Setup basic logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- Mock Tools ---
@tool
def search_web(query: str) -> str:
    """Searches the web for information based on the query."""
    logger.info(f"Tool Call: search_web with query: {query}")
    time.sleep(0.5) # Simulate network latency
    if "latest AI models" in query.lower():
        return "The latest AI models include GPT-5, Gemini Ultra, and Llama 4.0, focusing on multimodal capabilities and improved reasoning."
    return f"Mock search result for '{query}': Found some relevant articles."

@tool
def analyze_data(data: str) -> str:
    """Analyzes provided data to extract insights."""
    logger.info(f"Tool Call: analyze_data with data: {data}")
    time.sleep(0.3) # Simulate computation
    if "sales figures" in data.lower():
        return "Data analysis shows a 15% increase in Q3 sales compared to Q2."
    return f"Mock analysis result for '{data}': Insights extracted successfully."

# --- Agent State Definition ---
class AgentState(TypedDict):
    messages: List[BaseMessage]
    tool_calls: List[Dict]
    tool_output: str
    next: str # For supervisor routing

# --- Mock LLM for Agent Decisions ---
class MockLLM:
    def invoke(self, messages: List[BaseMessage], **kwargs) -> AIMessage:
        last_message = messages[-1].content.lower()
        logger.info(f"MockLLM invoked with last message: {last_message[:50]}...")
        
        # Simulate tool calling based on keywords
        if "search for" in last_message or "find information" in last_message:
            tool_call = {"name": "search_web", "args": {"query": last_message.replace("search for", "").strip()}}
            return AIMessage(content="", tool_calls=[tool_call])
        elif "analyze" in last_message or "data" in last_message:
            tool_call = {"name": "analyze_data", "args": {"data": last_message.replace("analyze", "").strip()}}
            return AIMessage(content="", tool_calls=[tool_call])
        elif "final answer" in last_message or "conclude" in last_message:
            return AIMessage(content=f"Final Answer: {last_message.replace('final answer:', '').strip()}")
        else:
            return AIMessage(content=f"I'm thinking about: {last_message}. What should I do next?")

# --- Agent Nodes ---
class AgentNode:
    def __init__(self, llm, tools, name):
        self.llm = llm
        self.tools = {t.name: t for t in tools}
        self.name = name

    def __call__(self, state: AgentState) -> AgentState:
        logger.info(f"Entering {self.name} node.")
        messages = state["messages"]
        response = self.llm.invoke(messages)
        
        new_messages = [response]
        tool_calls = response.tool_calls if hasattr(response, 'tool_calls') else []
        
        return {"messages": new_messages, "tool_calls": tool_calls}

# --- Tool Executor Node ---
def tool_executor_node(state: AgentState) -> AgentState:
    logger.info("Entering tool_executor_node.")
    tool_calls = state["tool_calls"]
    tool_output = ""
    if tool_calls:
        for tool_call in tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            
            # Execute the mock tool
            if tool_name == "search_web":
                output = search_web.invoke(tool_args)
            elif tool_name == "analyze_data":
                output = analyze_data.invoke(tool_args)
            else:
                output = f"Unknown tool: {tool_name}"
            
            tool_output += f"Tool '{tool_name}' output: {output}\n"
            logger.info(f"Executed tool {tool_name}, output: {output[:50]}...")
    
    return {"tool_output": tool_output, "messages": [AIMessage(content=tool_output)]}

# --- Supervisor Node ---
class Supervisor:
    def __init__(self, llm):
        self.llm = llm
        self.members = ["researcher", "analyst"]
        self.system_prompt = (
            "You are a supervisor agent. You oversee a team of specialized agents: researcher and analyst. "
            "Your goal is to direct the user's request to the most appropriate agent or to conclude the task. "
            "Respond with a single word: 'researcher', 'analyst', or 'FINISH'. "
            "Do not elaborate. If the task is complete or cannot be handled by the team, respond 'FINISH'."
        )

    def __call__(self, state: AgentState) -> Dict:
        logger.info("Entering supervisor node.")
        messages = [HumanMessage(content=self.system_prompt)] + state["messages"]
        response = self.llm.invoke(messages)
        
        # Simple keyword-based routing for mock supervisor
        content = response.content.lower()
        if "researcher" in content:
            next_agent = "researcher"
        elif "analyst" in content:
            next_agent = "analyst"
        else:
            next_agent = "FINISH"
            
        logger.info(f"Supervisor decided next agent: {next_agent}")
        return {"next": next_agent, "messages": [response]}

# --- Build the LangGraph Agent ---
def create_agent_pipeline():
    llm = MockLLM()
    tools = [search_web, analyze_data]

    researcher_agent = AgentNode(llm, [search_web], "researcher")
    analyst_agent = AgentNode(llm, [analyze_data], "analyst")
    supervisor = Supervisor(llm)

    workflow = StateGraph(AgentState)

    # Add nodes
    workflow.add_node("supervisor", supervisor)
    workflow.add_node("researcher", researcher_agent)
    workflow.add_node("analyst", analyst_agent)
    workflow.add_node("tool_executor", tool_executor_node)

    # Set entry point
    workflow.set_entry_point("supervisor")

    # Add edges
    workflow.add_edge(START, "supervisor")
    workflow.add_edge("researcher", "tool_executor")
    workflow.add_edge("analyst", "tool_executor")
    workflow.add_edge("tool_executor", "supervisor") # After tool execution, return to supervisor

    # Conditional edges from supervisor
    workflow.add_conditional_edges(
        "supervisor",
        lambda x: x["next"],
        {"researcher": "researcher", "analyst": "analyst", "FINISH": END}
    )

    # Compile the graph
    app = workflow.compile(checkpointer=MemorySaver())
    logger.info("LangGraph agent pipeline created successfully.")
    return app

# --- Dockerfile Structure (as a string) ---
dockerfile_content = """
# Use a lightweight Python base image
FROM python:3.10-slim-buster

# Set the working directory in the container
WORKDIR /app

# Copy the current directory contents into the container at /app
COPY . /app

# Install any needed packages specified in requirements.txt
RUN pip install --no-cache-dir "fastapi==0.110.0" "uvicorn==0.27.1" "langgraph==0.0.40" "langchain-core==0.1.30" "python-dotenv==1.0.1"

# Expose the port that FastAPI will run on
EXPOSE 8000

# Command to run the application
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

# --- main.py Structure (as a string, to be completed by student) ---
main_py_content_template = """
import os
import time
import logging
from typing import List, Dict, Any
from dotenv import load_dotenv

from fastapi import FastAPI, Request, HTTPException
from pydantic import BaseModel

# LangSmith integration (ensure LANGCHAIN_TRACING_V2=true and LANGCHAIN_API_KEY are set)
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY") # Make sure to set this in .env or env vars

# Load environment variables from .env file
load_dotenv()

# Import the agent creation function from the setup code (assuming it's in a separate file or copied here)
# For this exercise, we'll assume create_agent_pipeline is available in this scope.
# In a real scenario, you'd import it from a module.

# --- Mock Tools, Agent State, MockLLM, AgentNode, tool_executor_node, Supervisor, create_agent_pipeline ---
# (These would typically be imported from a separate module, but for this exercise,
# we'll assume they are defined or copied here for simplicity of a single file solution.)

# Copy the definitions from the setup cell above into this section for a self-contained main.py
# START COPY PASTE FROM SETUP CELL

{mock_agent_definitions}

# END COPY PASTE FROM SETUP CELL


app = FastAPI(title="Advanced LangGraph Agent API")

# Initialize the agent pipeline globally
agent_app = create_agent_pipeline()

# Request model for the API
class AgentQuery(BaseModel):
    query: str
    config: Dict[str, Any] = {}

@app.post("/invoke")
async def invoke_agent(agent_query: AgentQuery):
    start_time = time.time()
    query = agent_query.query
    config = agent_query.config

    logger.info(f"Received query: {query}")

    try:
        # --- STUDENT IMPLEMENTATION REQUIRED HERE ---
        # 1. Invoke the agent_app with the query and config.
        # 2. Capture the final output.
        # 3. Log relevant monitoring metrics (latency, conceptual token usage, state transitions).
        # 4. Return the agent's response.

        # Example placeholder for agent invocation:
        # final_state = agent_app.invoke({"messages": [HumanMessage(content=query)]}, config=config)
        # response_content = final_state["messages"][-1].content

        # --- END STUDENT IMPLEMENTATION ---

        end_time = time.time()
        latency = end_time - start_time
        logger.info(f"Agent run completed in {latency:.4f} seconds.")
        # Conceptual token usage (replace with actual LLM callback if available)
        conceptual_tokens = len(query.split()) * 2 + 50 # Rough estimate
        logger.info(f"Conceptual token usage for query '{query}': {conceptual_tokens} tokens.")

        return {"response": "Student needs to implement agent invocation and response extraction.", "latency": latency, "conceptual_tokens": conceptual_tokens}

    except Exception as e:
        logger.error(f"Error during agent invocation: {e}", exc_info=True)
        raise HTTPException(status_code=500, detail=f"Internal Server Error: {e}")

@app.get("/health")
async def health_check():
    return {"status": "ok", "agent_initialized": agent_app is not None}
"""

print("Setup complete. The following mock agent definitions and Dockerfile structure are ready.")
print("\n--- Mock Agent Definitions ---")
# This is just to show the student what's available, not to be executed directly.
# In the solution, these will be embedded into main.py or imported.
print(f"Mock LLM: {MockLLM.__name__}")
print(f"Mock Tools: {[t.name for t in [search_web, analyze_data]]}")
print(f"Agent State: {AgentState.__name__}")
print(f"Graph creation function: {create_agent_pipeline.__name__}")

print("\n--- Dockerfile Content ---")
print(dockerfile_content)

print("\n--- main.py Template (for student completion) ---")
# We'll print a truncated version to show the structure, the full content will be in the solution.
print(main_py_content_template.split('--- STUDENT IMPLEMENTATION REQUIRED HERE ---')[0] + "... (student implementation) ...")

# Store the mock agent definitions for later use in the solution
mock_agent_definitions_str = """
class MockLLM:
    def invoke(self, messages: List[BaseMessage], **kwargs) -> AIMessage:
        last_message = messages[-1].content.lower()
        logger.info(f"MockLLM invoked with last message: {last_message[:50]}...")
        
        if "search for" in last_message or "find information" in last_message:
            tool_call = {"name": "search_web", "args": {"query": last_message.replace("search for", "").strip()}}
            return AIMessage(content="", tool_calls=[tool_call])
        elif "analyze" in last_message or "data" in last_message:
            tool_call = {"name": "analyze_data", "args": {"data": last_message.replace("analyze", "").strip()}}
            return AIMessage(content="", tool_calls=[tool_call])
        elif "final answer" in last_message or "conclude" in last_message:
            return AIMessage(content=f"Final Answer: {last_message.replace('final answer:', '').strip()}")
        else:
            return AIMessage(content=f"I'm thinking about: {last_message}. What should I do next?")

@tool
def search_web(query: str) -> str:
    """Searches the web for information based on the query."""
    logger.info(f"Tool Call: search_web with query: {query}")
    time.sleep(0.5) # Simulate network latency
    if "latest AI models" in query.lower():
        return "The latest AI models include GPT-5, Gemini Ultra, and Llama 4.0, focusing on multimodal capabilities and improved reasoning."
    return f"Mock search result for '{query}': Found some relevant articles."

@tool
def analyze_data(data: str) -> str:
    """Analyzes provided data to extract insights."""
    logger.info(f"Tool Call: analyze_data with data: {data}")
    time.sleep(0.3) # Simulate computation
    if "sales figures" in data.lower():
        return "Data analysis shows a 15% increase in Q3 sales compared to Q2."
    return f"Mock analysis result for '{data}': Insights extracted successfully."

class AgentState(TypedDict):
    messages: List[BaseMessage]
    tool_calls: List[Dict]
    tool_output: str
    next: str # For supervisor routing

class AgentNode:
    def __init__(self, llm, tools, name):
        self.llm = llm
        self.tools = {t.name: t for t in tools}
        self.name = name

    def __call__(self, state: AgentState) -> AgentState:
        logger.info(f"Entering {self.name} node.")
        messages = state["messages"]
        response = self.llm.invoke(messages)
        
        new_messages = [response]
        tool_calls = response.tool_calls if hasattr(response, 'tool_calls') else []
        
        return {"messages": new_messages, "tool_calls": tool_calls}

def tool_executor_node(state: AgentState) -> AgentState:
    logger.info("Entering tool_executor_node.")
    tool_calls = state["tool_calls"]
    tool_output = ""
    if tool_calls:
        for tool_call in tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            
            if tool_name == "search_web":
                output = search_web.invoke(tool_args)
            elif tool_name == "analyze_data":
                output = analyze_data.invoke(tool_args)
            else:
                output = f"Unknown tool: {tool_name}"
            
            tool_output += f"Tool '{tool_name}' output: {output}\n"
            logger.info(f"Executed tool {tool_name}, output: {output[:50]}...")
    
    return {"tool_output": tool_output, "messages": [AIMessage(content=tool_output)]}

class Supervisor:
    def __init__(self, llm):
        self.llm = llm
        self.members = ["researcher", "analyst"]
        self.system_prompt = (
            "You are a supervisor agent. You oversee a team of specialized agents: researcher and analyst. "
            "Your goal is to direct the user's request to the most appropriate agent or to conclude the task. "
            "Respond with a single word: 'researcher', 'analyst', or 'FINISH'. "
            "Do not elaborate. If the task is complete or cannot be handled by the team, respond 'FINISH'."
        )

    def __call__(self, state: AgentState) -> Dict:
        logger.info("Entering supervisor node.")
        messages = [HumanMessage(content=self.system_prompt)] + state["messages"]
        response = self.llm.invoke(messages)
        
        content = response.content.lower()
        if "researcher" in content:
            next_agent = "researcher"
        elif "analyst" in content:
            next_agent = "analyst"
        else:
            next_agent = "FINISH"
            
        logger.info(f"Supervisor decided next agent: {next_agent}")
        return {"next": next_agent, "messages": [response]}

def create_agent_pipeline():
    llm = MockLLM()
    tools = [search_web, analyze_data]

    researcher_agent = AgentNode(llm, [search_web], "researcher")
    analyst_agent = AgentNode(llm, [analyze_data], "analyst")
    supervisor = Supervisor(llm)

    workflow = StateGraph(AgentState)

    workflow.add_node("supervisor", supervisor)
    workflow.add_node("researcher", researcher_agent)
    workflow.add_node("analyst", analyst_agent)
    workflow.add_node("tool_executor", tool_executor_node)

    workflow.set_entry_point("supervisor")

    workflow.add_edge(START, "supervisor")
    workflow.add_edge("researcher", "tool_executor")
    workflow.add_edge("analyst", "tool_executor")
    workflow.add_edge("tool_executor", "supervisor")

    workflow.add_conditional_edges(
        "supervisor",
        lambda x: x["next"],
        {"researcher": "researcher", "analyst": "analyst", "FINISH": END}
    )

    app = workflow.compile(checkpointer=MemorySaver())
    logger.info("LangGraph agent pipeline created successfully.")
    return app
"""


## Your Turn: Implement the Deployment and Monitoring Solution

Now it's your turn to complete the `main.py` file and prepare the `Dockerfile` for deployment. Your task is to:

1.  **Complete `main.py`**: Fill in the `invoke_agent` endpoint logic. You need to call the `agent_app.invoke` method with the user's query and configuration. Extract the final response from the agent's state.
2.  **Integrate Monitoring**: Ensure that the `latency` is correctly calculated and returned. Implement a conceptual `token_usage` calculation (as shown in the template) and ensure `logger.info` calls are strategically placed to track agent state transitions and tool calls.
3.  **LangSmith Tracing**: Uncomment and configure the `LANGCHAIN_TRACING_V2` and `LANGCHAIN_API_KEY` environment variables. If you have a LangSmith API key, use it. Otherwise, understand how it would be integrated.
4.  **Create `Dockerfile`**: Based on the provided `dockerfile_content` string in the setup cell, create a `Dockerfile` in your working directory. Ensure it correctly sets up the environment and runs your `main.py` application.
5.  **Test Locally**: Run your Docker container and interact with it using `curl` or a simple Python client to verify functionality, monitoring logs, and (if configured) LangSmith traces.

**Instructions:**

1.  Create a file named `main.py` in your current directory.
2.  Copy the `main_py_content_template` from the setup cell into `main.py`. **Crucially, replace the `START COPY PASTE FROM SETUP CELL` and `END COPY PASTE FROM SETUP CELL` block with the actual `mock_agent_definitions_str` content provided in the setup cell.** This will make your `main.py` self-contained.
3.  Implement the missing logic within the `invoke_agent` function in `main.py`.
4.  Create a file named `Dockerfile` in the same directory and paste the `dockerfile_content` into it.
5.  (Optional but recommended) Create a `.env` file with `LANGCHAIN_API_KEY=your_langsmith_api_key` if you want to test LangSmith integration.
6.  Build and run your Docker image:
    ```bash
    docker build -t advanced-agent-api .
    docker run -p 8000:8000 -e LANGCHAIN_TRACING_V2=true -e LANGCHAIN_API_KEY="your_langsmith_api_key" advanced-agent-api
    ```
    (Replace `your_langsmith_api_key` with your actual key or omit `-e LANGCHAIN_API_KEY` if not using LangSmith).
7.  Interact with your API using `curl` or the provided Python client example below.

```python
# Example Python Client (run in a separate terminal/script)
import requests
import json

api_url = "http://localhost:8000/invoke"
headers = {"Content-Type": "application/json"}

def send_query(query_text):
    payload = {"query": query_text, "config": {"configurable": {"session_id": "test_session_123"}}}
    try:
        response = requests.post(api_url, headers=headers, data=json.dumps(payload))
        response.raise_for_status() # Raise an exception for HTTP errors
        print(f"\nQuery: {query_text}")
        print(f"Response: {json.dumps(response.json(), indent=2)}")
    except requests.exceptions.RequestException as e:
        print(f"Error sending query: {e}")

if __name__ == "__main__":
    send_query("What are the latest AI models?")
    send_query("Analyze the sales figures for Q3.")
    send_query("Tell me a joke.") # Should lead to FINISH
```


In [ ]:
import os
import time
import logging
from typing import List, Dict, Any, TypedDict, Union, Tuple
from dotenv import load_dotenv

from fastapi import FastAPI, Request, HTTPException
from pydantic import BaseModel

# LangChain/LangGraph imports
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END, START
from langgraph.checkpoint.memory import MemorySaver

# Setup basic logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Load environment variables from .env file
load_dotenv()

# --- LangSmith Integration ---
# Ensure these are set for LangSmith tracing to work.
# LANGCHAIN_TRACING_V2=true
# LANGCHAIN_API_KEY=your_langsmith_api_key
# LANGCHAIN_PROJECT=your_project_name (optional, defaults to 'default')

# Set environment variables for LangSmith if not already set
# os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
# if os.getenv("LANGCHAIN_API_KEY"): # Only set if key is provided
#     os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
#     logger.info("LangSmith tracing enabled.")
# else:
#     logger.warning("LANGCHAIN_API_KEY not found. LangSmith tracing will be disabled.")

# --- Mock Tools ---
@tool
def search_web(query: str) -> str:
    """Searches the web for information based on the query."""
    logger.info(f"Tool Call: search_web with query: {query}")
    time.sleep(0.5) # Simulate network latency
    if "latest AI models" in query.lower():
        return "The latest AI models include GPT-5, Gemini Ultra, and Llama 4.0, focusing on multimodal capabilities and improved reasoning."
    return f"Mock search result for '{query}': Found some relevant articles."

@tool
def analyze_data(data: str) -> str:
    """Analyzes provided data to extract insights."""
    logger.info(f"Tool Call: analyze_data with data: {data}")
    time.sleep(0.3) # Simulate computation
    if "sales figures" in data.lower():
        return "Data analysis shows a 15% increase in Q3 sales compared to Q2."
    return f"Mock analysis result for '{data}': Insights extracted successfully."

# --- Agent State Definition ---
class AgentState(TypedDict):
    messages: List[BaseMessage]
    tool_calls: List[Dict]
    tool_output: str
    next: str # For supervisor routing

# --- Mock LLM for Agent Decisions ---
class MockLLM:
    def invoke(self, messages: List[BaseMessage], **kwargs) -> AIMessage:
        last_message = messages[-1].content.lower()
        logger.info(f"MockLLM invoked with last message: {last_message[:50]}...")
        
        # Simulate tool calling based on keywords
        if "search for" in last_message or "find information" in last_message or "latest ai models" in last_message:
            tool_call = {"name": "search_web", "args": {"query": last_message.replace("search for", "").strip()}}
            return AIMessage(content="", tool_calls=[tool_call])
        elif "analyze" in last_message or "data" in last_message or "sales figures" in last_message:
            tool_call = {"name": "analyze_data", "args": {"data": last_message.replace("analyze", "").strip()}}
            return AIMessage(content="", tool_calls=[tool_call])
        elif "final answer" in last_message or "conclude" in last_message or "joke" in last_message or "hello" in last_message:
            return AIMessage(content=f"Final Answer: {last_message.replace('final answer:', '').strip()}")
        else:
            return AIMessage(content=f"I'm thinking about: {last_message}. What should I do next?")

# --- Agent Nodes ---
class AgentNode:
    def __init__(self, llm, tools, name):
        self.llm = llm
        self.tools = {t.name: t for t in tools}
        self.name = name

    def __call__(self, state: AgentState) -> AgentState:
        logger.info(f"Entering {self.name} node. Current messages: {[m.content for m in state['messages']]}")
        messages = state["messages"]
        response = self.llm.invoke(messages)
        
        new_messages = [response]
        tool_calls = response.tool_calls if hasattr(response, 'tool_calls') else []
        
        return {"messages": new_messages, "tool_calls": tool_calls}

# --- Tool Executor Node ---
def tool_executor_node(state: AgentState) -> AgentState:
    logger.info("Entering tool_executor_node.")
    tool_calls = state["tool_calls"]
    tool_output = ""
    if tool_calls:
        for tool_call in tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            
            # Execute the mock tool
            if tool_name == "search_web":
                output = search_web.invoke(tool_args)
            elif tool_name == "analyze_data":
                output = analyze_data.invoke(tool_args)
            else:
                output = f"Unknown tool: {tool_name}"
            
            tool_output += f"Tool '{tool_name}' output: {output}\n"
            logger.info(f"Executed tool {tool_name}, output: {output[:50]}...")
    
    # Append tool output as an AI message to continue the conversation flow
    return {"tool_output": tool_output, "messages": [AIMessage(content=tool_output)]}

# --- Supervisor Node ---
class Supervisor:
    def __init__(self, llm):
        self.llm = llm
        self.members = ["researcher", "analyst"]
        self.system_prompt = (
            "You are a supervisor agent. You oversee a team of specialized agents: researcher and analyst. "
            "Your goal is to direct the user's request to the most appropriate agent or to conclude the task. "
            "Respond with a single word: 'researcher', 'analyst', or 'FINISH'. "
            "Do not elaborate. If the task is complete or cannot be handled by the team, respond 'FINISH'."
        )

    def __call__(self, state: AgentState) -> Dict:
        logger.info("Entering supervisor node.")
        messages = [HumanMessage(content=self.system_prompt)] + state["messages"]
        response = self.llm.invoke(messages)
        
        # Simple keyword-based routing for mock supervisor
        content = response.content.lower()
        if "researcher" in content or "search" in content or "find information" in content or "latest ai models" in content:
            next_agent = "researcher"
        elif "analyst" in content or "analyze" in content or "data" in content or "sales figures" in content:
            next_agent = "analyst"
        else:
            next_agent = "FINISH"
            
        logger.info(f"Supervisor decided next agent: {next_agent}")
        return {"next": next_agent, "messages": [response]}

# --- Build the LangGraph Agent ---
def create_agent_pipeline():
    llm = MockLLM()
    tools = [search_web, analyze_data]

    researcher_agent = AgentNode(llm, [search_web], "researcher")
    analyst_agent = AgentNode(llm, [analyze_data], "analyst")
    supervisor = Supervisor(llm)

    workflow = StateGraph(AgentState)

    # Add nodes
    workflow.add_node("supervisor", supervisor)
    workflow.add_node("researcher", researcher_agent)
    workflow.add_node("analyst", analyst_agent)
    workflow.add_node("tool_executor", tool_executor_node)

    # Set entry point
    workflow.set_entry_point("supervisor")

    # Add edges
    workflow.add_edge(START, "supervisor")
    workflow.add_edge("researcher", "tool_executor")
    workflow.add_edge("analyst", "tool_executor")
    workflow.add_edge("tool_executor", "supervisor") # After tool execution, return to supervisor

    # Conditional edges from supervisor
    workflow.add_conditional_edges(
        "supervisor",
        lambda x: x["next"],
        {"researcher": "researcher", "analyst": "analyst", "FINISH": END}
    )

    # Compile the graph
    app = workflow.compile(checkpointer=MemorySaver())
    logger.info("LangGraph agent pipeline created successfully.")
    return app


app = FastAPI(title="Advanced LangGraph Agent API")

# Initialize the agent pipeline globally
agent_app = create_agent_pipeline()

# Request model for the API
class AgentQuery(BaseModel):
    query: str
    config: Dict[str, Any] = {}

@app.post("/invoke")
async def invoke_agent(agent_query: AgentQuery):
    start_time = time.time()
    query = agent_query.query
    config = agent_query.config

    logger.info(f"Received query: {query}")

    try:
        # LangGraph expects messages in its state. Initialize with the human query.
        initial_state = {"messages": [HumanMessage(content=query)]}
        
        # Invoke the agent_app. The config dictionary can include session_id for checkpointer.
        # LangSmith tracing automatically picks up the config and session_id if set.
        final_state = agent_app.invoke(initial_state, config=config)
        
        # Extract the final response from the agent's state
        # The last message in the 'messages' list should be the final AI response.
        response_message = final_state["messages"][-1]
        response_content = response_message.content

        end_time = time.time()
        latency = end_time - start_time
        logger.info(f"Agent run completed in {latency:.4f} seconds.")
        
        # Conceptual token usage: A simple heuristic for demonstration.
        # In a real scenario, this would come from LLM callbacks or actual tokenizers.
        conceptual_tokens = len(query.split()) + len(response_content.split()) + 100 # Base + input + output
        logger.info(f"Conceptual token usage for query '{query}': {conceptual_tokens} tokens.")

        return {
            "response": response_content,
            "latency_seconds": latency,
            "conceptual_tokens": conceptual_tokens,
            "session_id": config.get("configurable", {}).get("session_id", "N/A")
        }

    except Exception as e:
        logger.error(f"Error during agent invocation: {e}", exc_info=True)
        raise HTTPException(status_code=500, detail=f"Internal Server Error: {e}")

@app.get("/health")
async def health_check():
    return {"status": "ok", "agent_initialized": agent_app is not None}

# --- Dockerfile Content ---
dockerfile_content = """
# Use a lightweight Python base image
FROM python:3.10-slim-buster

# Set the working directory in the container
WORKDIR /app

# Copy the current directory contents into the container at /app
COPY . /app

# Install any needed packages specified in requirements.txt
# Note: LangSmith integration requires 'langchain' or 'langchain-core' and setting env vars.
RUN pip install --no-cache-dir "fastapi==0.110.0" "uvicorn==0.27.1" "langgraph==0.0.40" "langchain-core==0.1.30" "python-dotenv==1.0.1"

# Expose the port that FastAPI will run on
EXPOSE 8000

# Command to run the application
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

print("Reference Solution: main.py and Dockerfile content provided below.")
print("\n--- To run this solution: ---")
print("1. Save the content of this cell (excluding the final print statements) into a file named `main.py`.")
print("2. Create a file named `Dockerfile` in the same directory and paste the `dockerfile_content` into it.")
print("3. (Optional) Create a `.env` file with `LANGCHAIN_API_KEY=your_langsmith_api_key` and `LANGCHAIN_TRACING_V2=true`.")
print("4. Build the Docker image: `docker build -t advanced-agent-api .`")
print("5. Run the Docker container: `docker run -p 8000:8000 -e LANGCHAIN_TRACING_V2=true -e LANGCHAIN_API_KEY="your_langsmith_api_key" advanced-agent-api`")
print("   (Replace `your_langsmith_api_key` with your actual key or omit `-e LANGCHAIN_API_KEY` if not using LangSmith).")
print("6. Use the Python client or `curl` to interact with the API at `http://localhost:8000/invoke`.")

print("\n--- Example Python Client (run in a separate terminal/script) ---")
print("""
import requests
import json

api_url = "http://localhost:8000/invoke"
headers = {"Content-Type": "application/json"}

def send_query(query_text, session_id="test_session_123"):
    payload = {"query": query_text, "config": {"configurable": {"session_id": session_id}}}
    try:
        response = requests.post(api_url, headers=headers, data=json.dumps(payload))
        response.raise_for_status() # Raise an exception for HTTP errors
        print(f"\nQuery: {query_text}")
        print(f"Response: {json.dumps(response.json(), indent=2)}")
    except requests.exceptions.RequestException as e:
        print(f"Error sending query: {e}")

if __name__ == "__main__":
    send_query("What are the latest AI models?")
    send_query("Analyze the sales figures for Q3.")
    send_query("Tell me a joke.") # Should lead to FINISH
    send_query("Hello there!", session_id="greeting_session")
""")
